# Spotify + YouTube Dataset → Cassandra

**Goal:** Clean the CSV and save every remaining column in one Cassandra table.

- Source file: `Spotify_Youtube(1).csv`
- Table: `spotify.spotify_youtube_full`
- Missing values: remove every row that has at least one missing value.
- Run each cell from top to bottom.

## 1. Install the required libraries
Run this cell once if the libraries are not installed.

In [4]:
%pip install pandas cassandra-driver

Note: you may need to restart the kernel to use updated packages.


## 2. Import libraries and set the CSV path

In [5]:
import pandas as pd
from cassandra.cluster import Cluster

CSV_FILE = "Spotify_Youtube.csv"

# Cassandra connection settings
CASSANDRA_HOST = "127.0.0.1"
CASSANDRA_PORT = 9042

## 3. Load the CSV and check its size

In [6]:
df = pd.read_csv(CSV_FILE)

print("Original rows and columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
df.head()

Original rows and columns: (20718, 28)

Columns:
['Unnamed: 0', 'Artist', 'Url_spotify', 'Track', 'Album', 'Album_type', 'Uri', 'Danceability', 'Energy', 'Key', 'Loudness', 'Speechiness', 'Acousticness', 'Instrumentalness', 'Liveness', 'Valence', 'Tempo', 'Duration_ms', 'Url_youtube', 'Title', 'Channel', 'Views', 'Likes', 'Comments', 'Description', 'Licensed', 'official_video', 'Stream']


,Unnamed: 0,Artist,Url_spotify,Track,Album,Album_type,Uri,Danceability,Energy,Key,...,Url_youtube,Title,Channel,Views,Likes,Comments,Description,Licensed,official_video,Stream
0,0,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Feel Good Inc.,Demon Days,album,spotify:track:0d28khcov6AiegSCpG5TuT,0.818,0.705,6.0,...,https://www.youtube.com/watch?v=HyHNuVaZJ-k,Gorillaz - Feel Good Inc. (Official Video),Gorillaz,693555221.0,6220896.0,169907.0,Official HD Video for Gorillaz' fantastic trac...,True,True,1.040235e+09
1,1,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Rhinestone Eyes,Plastic Beach,album,spotify:track:1foMv2HQwfQ2vntFf9HFeG,0.676,0.703,8.0,...,https://www.youtube.com/watch?v=yYDmaexVHic,Gorillaz - Rhinestone Eyes [Storyboard Film] (...,Gorillaz,72011645.0,1079128.0,31003.0,The official video for Gorillaz - Rhinestone E...,True,True,3.100837e+08
2,2,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,spotify:track:64dLd6rVqDLtkXFYrEUHIU,0.695,0.923,1.0,...,https://www.youtube.com/watch?v=qJa-VFwPpYA,Gorillaz - New Gold ft. Tame Impala & Bootie B...,Gorillaz,8435055.0,282142.0,7399.0,Gorillaz - New Gold ft. Tame Impala & Bootie B...,True,True,6.306347e+07
3,3,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,On Melancholy Hill,Plastic Beach,album,spotify:track:0q6LuUqGLUiCPP1cbdwFs3,0.689,0.739,2.0,...,https://www.youtube.com/watch?v=04mfKJWDSzI,Gorillaz - On Melancholy Hill (Official Video),Gorillaz,211754952.0,1788577.0,55229.0,Follow Gorillaz online:\nhttp://gorillaz.com \...,True,True,4.346636e+08
4,4,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Clint Eastwood,Gorillaz,album,spotify:track:7yMiX7n9SBvadzox8T5jzT,0.663,0.694,10.0,...,https://www.youtube.com/watch?v=1V_xRb0x9aw,Gorillaz - Clint Eastwood (Official Video),Gorillaz,618480958.0,6197318.0,155930.0,The official music video for Gorillaz - Clint ...,True,True,6.172597e+08


## 4. Remove missing values and the extra index column

In [7]:
# 'Unnamed: 0' is only a CSV row index, not a Spotify feature.
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Remove rows with any missing value.
df = df.dropna().copy()

# Give each retained row a simple unique ID for Cassandra.
df.insert(0, "row_id", range(1, len(df) + 1))

print("Rows after removing missing values:", len(df))
print("Remaining missing values:", int(df.isna().sum().sum()))
df.head()

Rows after removing missing values: 19170
Remaining missing values: 0


,row_id,Artist,Url_spotify,Track,Album,Album_type,Uri,Danceability,Energy,Key,...,Url_youtube,Title,Channel,Views,Likes,Comments,Description,Licensed,official_video,Stream
0,1,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Feel Good Inc.,Demon Days,album,spotify:track:0d28khcov6AiegSCpG5TuT,0.818,0.705,6.0,...,https://www.youtube.com/watch?v=HyHNuVaZJ-k,Gorillaz - Feel Good Inc. (Official Video),Gorillaz,693555221.0,6220896.0,169907.0,Official HD Video for Gorillaz' fantastic trac...,True,True,1.040235e+09
1,2,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Rhinestone Eyes,Plastic Beach,album,spotify:track:1foMv2HQwfQ2vntFf9HFeG,0.676,0.703,8.0,...,https://www.youtube.com/watch?v=yYDmaexVHic,Gorillaz - Rhinestone Eyes [Storyboard Film] (...,Gorillaz,72011645.0,1079128.0,31003.0,The official video for Gorillaz - Rhinestone E...,True,True,3.100837e+08
2,3,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,spotify:track:64dLd6rVqDLtkXFYrEUHIU,0.695,0.923,1.0,...,https://www.youtube.com/watch?v=qJa-VFwPpYA,Gorillaz - New Gold ft. Tame Impala & Bootie B...,Gorillaz,8435055.0,282142.0,7399.0,Gorillaz - New Gold ft. Tame Impala & Bootie B...,True,True,6.306347e+07
3,4,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,On Melancholy Hill,Plastic Beach,album,spotify:track:0q6LuUqGLUiCPP1cbdwFs3,0.689,0.739,2.0,...,https://www.youtube.com/watch?v=04mfKJWDSzI,Gorillaz - On Melancholy Hill (Official Video),Gorillaz,211754952.0,1788577.0,55229.0,Follow Gorillaz online:\nhttp://gorillaz.com \...,True,True,4.346636e+08
4,5,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Clint Eastwood,Gorillaz,album,spotify:track:7yMiX7n9SBvadzox8T5jzT,0.663,0.694,10.0,...,https://www.youtube.com/watch?v=1V_xRb0x9aw,Gorillaz - Clint Eastwood (Official Video),Gorillaz,618480958.0,6197318.0,155930.0,The official music video for Gorillaz - Clint ...,True,True,6.172597e+08


## 5. Clean column names
Cassandra column names will be lowercase and use underscores.

In [8]:
df.columns = [
    col.strip().lower().replace(" ", "_")
    for col in df.columns
]

print(df.columns.tolist())

['row_id', 'artist', 'url_spotify', 'track', 'album', 'album_type', 'uri', 'danceability', 'energy', 'key', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'url_youtube', 'title', 'channel', 'views', 'likes', 'comments', 'description', 'licensed', 'official_video', 'stream']


## 6. Convert values to Cassandra-friendly types

In [9]:
# These columns contain True/False values in the CSV.
for column in ["licensed", "official_video"]:
    df[column] = df[column].astype(bool)

# Convert numeric features and counts to numbers.
numeric_columns = [
    "danceability", "energy", "key", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "duration_ms", "views", "likes", "comments", "stream"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column])

print(df.dtypes)

row_id                int64
artist                  str
url_spotify             str
track                   str
album                   str
album_type              str
uri                     str
danceability        float64
energy              float64
key                 float64
loudness            float64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
duration_ms         float64
url_youtube             str
title                   str
channel                 str
views               float64
likes               float64
comments            float64
description             str
licensed               bool
official_video         bool
stream              float64
dtype: object


## 7. Connect to Cassandra and create the keyspace

In [10]:
cluster = Cluster([CASSANDRA_HOST], port=CASSANDRA_PORT)
session = cluster.connect()

session.execute("""
CREATE KEYSPACE IF NOT EXISTS spotify
WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}
""")

session.set_keyspace("spotify")
print("Connected to Cassandra. Keyspace 'spotify' is ready.")

Connected to Cassandra. Keyspace 'spotify' is ready.


## 8. Create one table for all columns
This table contains song details, all audio features, Spotify streams, YouTube views/likes/comments, links, and metadata.

In [11]:
session.execute("""
CREATE TABLE IF NOT EXISTS spotify_youtube_full (
    row_id INT PRIMARY KEY,
    artist TEXT,
    url_spotify TEXT,
    track TEXT,
    album TEXT,
    album_type TEXT,
    uri TEXT,
    danceability DOUBLE,
    energy DOUBLE,
    key DOUBLE,
    loudness DOUBLE,
    speechiness DOUBLE,
    acousticness DOUBLE,
    instrumentalness DOUBLE,
    liveness DOUBLE,
    valence DOUBLE,
    tempo DOUBLE,
    duration_ms DOUBLE,
    url_youtube TEXT,
    title TEXT,
    channel TEXT,
    views DOUBLE,
    likes DOUBLE,
    comments DOUBLE,
    description TEXT,
    licensed BOOLEAN,
    official_video BOOLEAN,
    stream DOUBLE
)
""")

print("Table 'spotify.spotify_youtube_full' is ready.")

Table 'spotify.spotify_youtube_full' is ready.


## 9. Prepare the insert query

In [12]:
insert_query = session.prepare("""
INSERT INTO spotify_youtube_full (
    row_id, artist, url_spotify, track, album, album_type, uri,
    danceability, energy, key, loudness, speechiness, acousticness,
    instrumentalness, liveness, valence, tempo, duration_ms,
    url_youtube, title, channel, views, likes, comments, description,
    licensed, official_video, stream
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

print("Insert query prepared.")

Insert query prepared.


## 10. Save the cleaned rows to Cassandra
This inserts one row at a time, which keeps the code easy to follow. If you rerun the notebook, the same `row_id` values are updated rather than duplicated.

In [13]:
columns = [
    "row_id", "artist", "url_spotify", "track", "album", "album_type", "uri",
    "danceability", "energy", "key", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration_ms",
    "url_youtube", "title", "channel", "views", "likes", "comments", "description",
    "licensed", "official_video", "stream"
]

saved = 0

for row in df[columns].itertuples(index=False, name=None):
    session.execute(insert_query, row)
    saved += 1

print(f"Rows sent to Cassandra: {saved}")

Rows sent to Cassandra: 19170


## 11. Verify the saved data

In [14]:
count_row = session.execute(
    "SELECT COUNT(*) FROM spotify_youtube_full"
).one()

print("Rows currently in Cassandra:", count_row.count)

sample = session.execute("""
SELECT artist, track, danceability, energy, liveness,
       views, likes, comments, stream
FROM spotify_youtube_full LIMIT 5
""")

for song in sample:
    print(song)

Rows currently in Cassandra: 19170
Row(artist='Sheryl Crow', track='Strong Enough', danceability=0.58, energy=0.265, liveness=0.11, views=97038.0, likes=1798.0, comments=102.0, stream=51767995.0)
Row(artist='Etta James', track='Trust In Me', danceability=0.486, energy=0.377, liveness=0.39, views=985595.0, likes=7942.0, comments=185.0, stream=17660226.0)
Row(artist='Mon Laferte', track='Amor Completo', danceability=0.393, energy=0.418, liveness=0.108, views=170138206.0, likes=677915.0, comments=14805.0, stream=129999337.0)
Row(artist='Apache 207', track='Nie mehr gehen', danceability=0.704, energy=0.318, liveness=0.17, views=2293294.0, likes=58206.0, comments=3092.0, stream=9903711.0)
Row(artist='London Symphony Orchestra', track='Duel of the Fates', danceability=0.471, energy=0.411, liveness=0.28, views=259284.0, likes=4134.0, comments=285.0, stream=91702045.0)


## 12. Close the connection

In [15]:
cluster.shutdown()
print("Cassandra connection closed.")

Cassandra connection closed.


## Notes

- `dropna()` removes a row if **any** column is missing. This means the final dataset will contain fewer rows than the original CSV.
- `row_id` is generated after cleaning and is the primary key for this table.
- `CREATE TABLE IF NOT EXISTS` does not replace an existing table schema. If a previous table with the same name already exists with different columns, inspect it before running this notebook.
- This notebook expects Cassandra to be running locally at `127.0.0.1:9042`.